In [1]:
from transformers import (
    Wav2Vec2Model,
    Wav2Vec2BertModel,
    AutoProcessor,
    AutoFeatureExtractor,
)
import torch

# Load Wav2Vec2 model and processor
wav2vec_model = Wav2Vec2Model.from_pretrained(
    "facebook/wav2vec2-base-960h", torch_dtype=torch.bfloat16, device_map="cuda"
)
wav2vec_processor = AutoProcessor.from_pretrained("facebook/wav2vec2-base-960h")

# Load Wav2Vec2-BERT model and feature extractor
wav2vec_bert_model = Wav2Vec2BertModel.from_pretrained(
    "facebook/w2v-bert-2.0", torch_dtype=torch.bfloat16, device_map="cuda"
)
wav2vec_bert_processor = AutoFeatureExtractor.from_pretrained("facebook/w2v-bert-2.0")

# Optional: Set models to evaluation mode
wav2vec_model.eval()
wav2vec_bert_model.eval()

Some weights of Wav2Vec2Model were not initialized from the model checkpoint at facebook/wav2vec2-base-960h and are newly initialized: ['wav2vec2.masked_spec_embed']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Wav2Vec2BertModel(
  (feature_projection): Wav2Vec2BertFeatureProjection(
    (layer_norm): LayerNorm((160,), eps=1e-05, elementwise_affine=True)
    (projection): Linear(in_features=160, out_features=1024, bias=True)
    (dropout): Dropout(p=0.0, inplace=False)
  )
  (encoder): Wav2Vec2BertEncoder(
    (dropout): Dropout(p=0.0, inplace=False)
    (layers): ModuleList(
      (0-23): 24 x Wav2Vec2BertEncoderLayer(
        (ffn1_layer_norm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
        (ffn1): Wav2Vec2BertFeedForward(
          (intermediate_dropout): Dropout(p=0.0, inplace=False)
          (intermediate_dense): Linear(in_features=1024, out_features=4096, bias=True)
          (intermediate_act_fn): SiLU()
          (output_dense): Linear(in_features=4096, out_features=1024, bias=True)
          (output_dropout): Dropout(p=0.0, inplace=False)
        )
        (self_attn_layer_norm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
        (self_attn_dropout): Drop

In [2]:
from datasets import load_dataset

dataset = load_dataset(
    "hf-internal-testing/librispeech_asr_demo", "clean", split="validation"
)
dataset = dataset.sort("id")
sampling_rate = dataset.features["audio"].sampling_rate
audio_array = dataset[0]["audio"]["array"]


import numpy as np


def pad_audio_to_30_seconds(audio_array, sampling_rate):
    """
    Pad or trim audio array to exactly 30 seconds.

    Args:
        audio_array (np.ndarray): Input audio array
        sampling_rate (int): Sampling rate of the audio

    Returns:
        np.ndarray: Padded/trimmed audio array of exactly 30 seconds
    """
    target_length = 30 * sampling_rate  # 30 seconds * sampling_rate
    current_length = len(audio_array)

    if current_length < target_length:
        # Pad with zeros if audio is shorter than 30 seconds
        padding_length = target_length - current_length
        padded_array = np.pad(audio_array, (0, padding_length), mode="constant")
        return padded_array
    else:
        # Trim if audio is longer than 30 seconds
        return audio_array[:target_length]


# Apply the padding to your audio
padded_audio = pad_audio_to_30_seconds(audio_array, sampling_rate)

# Verify the length
print(f"Original duration: {len(audio_array) / sampling_rate:.2f} seconds")
print(f"Padded duration: {len(padded_audio) / sampling_rate:.2f} seconds")

Original duration: 5.86 seconds
Padded duration: 30.00 seconds


In [3]:
wav2vec_inputs = wav2vec_processor(
    padded_audio, sampling_rate=sampling_rate, return_tensors="pt"
)
wav2vec_inputs = {k: v.to("cuda", torch.bfloat16) for k, v in wav2vec_inputs.items()}

In [4]:
with torch.no_grad():
    wav2vec_outputs = wav2vec_model(**wav2vec_inputs)

# print
print(
    "Shape of the last hidden state for wav2vec_model:",
    wav2vec_outputs.last_hidden_state.shape,
)
print(
    "Shape of extract_features output for wav2vec_model:",
    wav2vec_outputs.extract_features.shape,
)

Shape of the last hidden state for wav2vec_model: torch.Size([1, 1499, 768])
Shape of extract_features output for wav2vec_model: torch.Size([1, 1499, 512])


In [5]:
wav2vec_bert_inputs = wav2vec_bert_processor(
    padded_audio, sampling_rate=sampling_rate, return_tensors="pt"
)
wav2vec_bert_inputs = {
    "input_features": wav2vec_bert_inputs["input_features"].cuda().to(torch.bfloat16),
    "attention_mask": wav2vec_bert_inputs["attention_mask"].cuda().to(torch.int32),
}


with torch.no_grad():
    wav2vec_bert_outputs = wav2vec_bert_model(**wav2vec_bert_inputs)

print(
    "Shape of the last hidden state for wav2vec_bert_model:",
    wav2vec_bert_outputs.last_hidden_state.shape,
)
print(
    "Shape of extract_features output for wav2vec_bert_model:",
    wav2vec_bert_outputs.extract_features.shape,
)

Shape of the last hidden state for wav2vec_bert_model: torch.Size([1, 1499, 1024])
Shape of extract_features output for wav2vec_bert_model: torch.Size([1, 1499, 160])


In [6]:
print(
    "Shape of input features for wav2vec_bert_model:",
    wav2vec_bert_inputs["input_features"].shape,
)

Shape of input features for wav2vec_bert_model: torch.Size([1, 1499, 160])


In [7]:
from transformers import WhisperModel, WhisperProcessor

# Load Whisper model and processor
whisper_model = WhisperModel.from_pretrained(
    "openai/whisper-large-v3", torch_dtype=torch.bfloat16, device_map="cuda"
)
whisper_model.eval()
whisper_processor = WhisperProcessor.from_pretrained("openai/whisper-large-v3")

# Process audio input
whisper_inputs = whisper_processor(
    padded_audio, sampling_rate=sampling_rate, return_tensors="pt"
)
whisper_inputs = {k: v.to("cuda", torch.bfloat16) for k, v in whisper_inputs.items()}

# Get encoder outputs
with torch.no_grad():
    whisper_outputs = whisper_model.get_encoder()(**whisper_inputs)

# Print shapes of encoder outputs
print(
    "Shape of the encoder last hidden state for whisper-large-v3:",
    whisper_outputs.last_hidden_state.shape,
)

Shape of the encoder last hidden state for whisper-large-v3: torch.Size([1, 1500, 1280])
